# Fairness Over Time — Phase 4: Fix (Texas, fit on 2016, test on 2017)

**RQ3: Which maintenance strategy keeps the model fair most cheaply?**

Phase 3 found that the problem sits at the **decision threshold** (the score cutoff that turns a probability into yes/no), not in the ranking: AUC stays flat, but the 2014 threshold no longer fits a population whose incomes rose. This notebook compares five strategies, from doing nothing to retraining with fairness constraints:

| Code | Strategy | What changes | Uses 2016 labels? |
|---|---|---|---|
| S0 | **Do nothing** (`none`, `none_mitigated`) | Nothing: the frozen 2014 models | No |
| S1 | **One new cutoff** (`global_cutoff`) | One decision threshold for everyone | Yes |
| S2 | **Group cutoffs** (`group_cutoffs`) | One threshold per race/ethnicity × sex group (Fairlearn `ThresholdOptimizer`) | Yes |
| S3 | **Retrain** (`retrain`) | New preprocessor + model, same settings | Yes |
| S4 | **Retrain + re-mitigate** (`retrain_mitigate`) | New model + `ExponentiatedGradient` with equalized odds | Yes |

Each strategy is judged on **fairness** (EOD, DPD), **accuracy** and **cost** (fitting time, data needed, and how hard it is to run in practice).

In [1]:
%load_ext autoreload
%autoreload 2

import time
import warnings
import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import accuracy_score, roc_auc_score

from src.data import DATA_DIR, ARTIFACTS_DIR, load_acs, make_race_eth, split_data, save_split, load_split
from src.preprocess import NUMERIC_COLS, ORDINAL_COLS, NOMINAL_COLS, build_preprocessor
from src.models import make_lr, make_xgb, make_pipeline_for, fit_mitigated, load_models, MODEL_INFO
from src.fairness import (make_sensitive, group_metrics, fairness_summary, fairness_long,
                          bootstrap_gaps, evaluate_all, bootstrap_all, save_results)

warnings.filterwarnings("ignore", module="fairlearn")

SEED = 42

## Setup: the scenario

Every fix is **fitted on 2016 and evaluated on all of 2017**. This mimics real life: a team notices drift, uses the latest year with known outcomes to fix the model, and the fix is then used on the next year.

Fitting on 2016 is a best case: ACS data is published about a year after the survey year, so a real team would only have 2015 labels when making 2017 decisions. Fitting on 2017 itself would be leakage.

After unpacking, the generic `X, y, meta` names are deleted, so no later cell can fit on the test year by accident. The four frozen models are loaded as in Phases 2 and 3; they are never changed (new models are built from copies).

In [2]:
#load data
data = {}
for year in [2016, 2017]:
    X, y, meta = load_acs(year, "TX")
    y = y.squeeze("columns")
    assert X.isna().sum().sum() == 0
    assert len(X) == len(y) == len(meta)

    data[year] = (X, y, meta)
    print(f"{year}: {len(X):,} rows | positive rate {y.mean():.4f} | y type: {type(y).__name__}")

2016: 131,089 rows | positive rate 0.3415 | y type: Series
2017: 133,880 rows | positive rate 0.3539 | y type: Series


In [3]:
X16, y16, meta16 = data[2016]
X17, y17, meta17 = data[2017]

del X, y, meta

In [4]:
#load 4 models
models = load_models()

models.keys()

dict_keys(['lr', 'lr_mitigated', 'xgb', 'xgb_mitigated'])

## S0: Do nothing (baseline)

The frozen 2014 models evaluated on 2017, both unmitigated (`none`) and mitigated in 2014 (`none_mitigated`). Every other strategy is compared with these rows. Fitting time is 0 and no new data is needed.

Results are converted to the Phase 4 contract with `to_fix_rows()`: `strategy, model, attribute, group, metric, value`, plus one `fit_seconds` row per strategy and model (94 rows each).

In [5]:
from src.fairness import evaluate_model, to_fix_rows
from src.data import METRICS_PATH, FIX_PATH

S0 = {"lr": "none", "xgb": "none",
      "lr_mitigated": "none_mitigated", "xgb_mitigated": "none_mitigated"}

fix_rows = []
for name, strategy in S0.items():
    ev = evaluate_model(models[name], X17, y17, meta17)
    label = MODEL_INFO[name]["model"]
    fix_rows.append(to_fix_rows(ev, strategy, label, fit_seconds=0))

s0 = pd.concat(fix_rows, ignore_index=True)
print(len(s0))   # expect 4 × 93 + 4 = 376
s0.groupby(["strategy", "model"]).size()

376


strategy        model  
none            LR         94
                XGBoost    94
none_mitigated  LR         94
                XGBoost    94
dtype: int64

In [6]:
ref = pd.read_csv(METRICS_PATH).query("year == 2017")

new = (s0[s0["metric"] != "fit_seconds"]
       .assign(mitigated=lambda d: d["strategy"].eq("none_mitigated")))

check = new.merge(ref, on=["model", "mitigated", "attribute", "group", "metric"],
                  how="outer", suffixes=("_new", "_ref"), indicator=True)

print(check["_merge"].value_counts())          # want: both = 372, nothing else
diff = (check["value_new"] - check["value_ref"]).abs()
print("max diff:", diff.max())
check.loc[diff > 1e-9, ["model", "mitigated", "attribute", "group", "metric",
                        "value_new", "value_ref"]]

_merge
both          372
left_only       0
right_only      0
Name: count, dtype: int64
max diff: 0.0


,model,mitigated,attribute,group,metric,value_new,value_ref


**Check passes:** S0 reproduces the 2017 rows of `metrics_over_time.csv` from Phase 2 exactly (372 of 372 rows matched, maximum difference 0.0). So Phase 4 uses the same models, data and metrics as Phase 2, and any difference below comes from the strategy, not the code.

## S1: One new cutoff for everyone

Keep the model and its scores, and only move the decision threshold. The threshold is chosen to **maximize accuracy on 2016**, searching 0.05 to 0.95 in steps of 0.005. `GlobalThresholdClassifier` wraps the frozen model: same scores, new threshold.

This is the cheapest fix that uses new data, and it tests Phase 3's idea directly: if the problem is an out-of-date threshold, moving it should help.

In [7]:
from src.models import GlobalThresholdClassifier

GRID = np.round(np.arange(0.05, 0.95, 0.005), 3)
y16_np = y16.to_numpy().astype(int)
s1_info = {}

for name in ["lr", "xgb"]:
    base = models[name]
    label = MODEL_INFO[name]["model"]

    t0 = time.perf_counter()
    s16 = base.predict_proba(X16)[:, 1]                      # 2016 scores, computed once
    accs = np.array([((s16 >= t) == y16_np).mean() for t in GRID])
    best_t = GRID[accs.argmax()]
    fit_seconds = time.perf_counter() - t0

    acc_05 = ((s16 >= 0.5) == y16_np).mean()                 # check vs Phase 2 2016
    s1_info[label] = {"threshold": best_t, "acc16_at_0.5": acc_05,
                      "acc16_best": accs.max(), "fit_seconds": fit_seconds}

    clf = GlobalThresholdClassifier(base, best_t)
    ev = evaluate_model(clf, X17, y17, meta17)
    fix_rows.append(to_fix_rows(ev, "global_cutoff", label, fit_seconds))

pd.DataFrame(s1_info).T.round(4)

,threshold,acc16_at_0.5,acc16_best,fit_seconds
LR,0.45,0.8127,0.8152,0.3779
XGBoost,0.46,0.8191,0.8199,0.3558


In [8]:
s1 = pd.concat(fix_rows, ignore_index=True).query("strategy == 'global_cutoff'")
s0 = pd.concat(fix_rows, ignore_index=True).query("strategy == 'none'")

# AUC must be identical (same scores)
auc = lambda d: d.query("attribute == 'overall' and metric == 'auc'").set_index("model")["value"]
print("AUC S1 - S0:", (auc(s1) - auc(s0)).abs().max())       # want 0.0

# Headline comparison on 2017
key = lambda d, m: d.query("(attribute == 'overall' and metric == @m) or "
                           "(attribute == 'race_eth_x_sex' and metric == @m)").set_index("model")["value"]
pd.DataFrame({m: key(s1, m) - key(s0, m) for m in ["accuracy", "DPD", "EOD"]}).round(4)

AUC S1 - S0: 0.0


,accuracy,DPD,EOD
model,,,
LR,0.0027,0.0085,-0.0450
XGBoost,0.0022,0.0101,-0.0253


### Finding: S1 helps a little, mostly by undoing drift

- **The best 2016 threshold is 0.45 (LR) and 0.46 (XGBoost),** below 0.5 as expected: the same score now goes with a higher chance of earning over $50K, so the line should come down.
- **Accuracy recovers only about a fifth of the 2014 → 2017 drop** (+0.27 points LR, +0.22 XGBoost). The accuracy curve is flat near its best point, so moving the threshold gains little.
- **EOD falls by 0.045 (LR) and 0.025 (XGBoost),** back to about the 2014 level (0.533 vs. 0.524 in 2014; 0.482 vs. 0.469). Lowering the threshold lifts the low-scored groups whose newly qualified members sat just below it. LR's change is just above noise; XGBoost's is within noise.
- DPD rises by about 0.01 (noise). AUC is identical to S0, as it must be: the scores are unchanged (check: difference 0.0).

**S1 undoes the part of the unfairness caused by drift, but not the unfairness the model already had in 2014.**

## S2: A separate cutoff for each group

Fairlearn's `ThresholdOptimizer` keeps the model's scores but picks a **different decision threshold for each of the 10 race/ethnicity × sex groups**, chosen on 2016 to maximize accuracy **subject to equalized odds** (TPR and FPR as equal as possible across groups).

- Wrapped in `GroupThresholdClassifier`, which reads sex and race/ethnicity from `X` to know which threshold to use.
- **Predictions are randomized** for people who fall between two thresholds (seed 42), because that is how ThresholdOptimizer reaches equal rates; this is explained with the rules table below.
- Two compatibility fixes were needed: XGBoost returns float32 scores, which ThresholdOptimizer rejects, and newer scikit-learn requires extra "tags". Both are handled by a small adapter in `src/models.py`.

In [9]:
from src.models import GroupThresholdClassifier

s2_fitted = {}
for name in ["lr", "xgb"]:
    label = MODEL_INFO[name]["model"]

    t0 = time.perf_counter()
    clf = GroupThresholdClassifier(models[name], seed=SEED).fit(X16, y16)
    fit_seconds = time.perf_counter() - t0

    ev = evaluate_model(clf, X17, y17, meta17)
    fix_rows.append(to_fix_rows(ev, "group_cutoffs", label, fit_seconds))
    s2_fitted[label] = clf
    print(f"{label}: fit {fit_seconds:.1f}s")

LR: fit 1.6s
XGBoost: fit 1.5s


In [10]:
def rules_table(clf):
    d = clf.to_.interpolated_thresholder_.interpolation_dict
    rows = []
    for g, v in d.items():
        rows.append({"group": g,
                     "p0": v["p0"], "cutoff0": v["operation0"].threshold,
                     "p1": v["p1"], "cutoff1": v["operation1"].threshold})
    return pd.DataFrame(rows).round(3)

rules_table(s2_fitted["LR"])

,group,p0,cutoff0,p1,cutoff1
0,"1.0,Hispanic",0.936,0.280,0.064,0.268
1,"1.0,NH Asian",0.367,0.671,0.633,0.615
2,"1.0,NH Black",0.995,0.379,0.005,0.331
3,"1.0,NH Other",0.428,0.434,0.572,0.401
4,"1.0,NH White",0.708,0.619,0.292,0.608
5,"2.0,Hispanic",0.554,0.170,0.446,0.159
6,"2.0,NH Asian",0.263,0.513,0.737,0.371
7,"2.0,NH Black",0.739,0.243,0.261,0.230
8,"2.0,NH Other",0.921,0.285,0.079,0.189
9,"2.0,NH White",0.203,0.451,0.797,0.390


### How to read the rules

Each group gets two thresholds and two weights:
- Score above the **higher** threshold → always "yes".
- Score below the **lower** threshold → always "no".
- Score in between → "yes" with a fixed probability set by `p0` / `p1`. That is the randomized part.

In the table, SEX is coded 1 = Male, 2 = Female. The **effective** threshold for LR ranges from about **0.16 for Hispanic women** (0.17 / 0.16), 0.24 for NH Black women and 0.40 for NH White women, up to about **0.61 for NH White men** and 0.64 for NH Asian men: roughly a 4× spread, compared with S1's single 0.45.

Some groups are almost deterministic: NH Black men have p0 = 0.995, so effectively one threshold of 0.379.

**What this means:** to give every group a similar TPR, the low-scored groups need a much lower bar. This is exactly the gap Phase 3 found, closed by force at the decision step.

In [11]:
fix_df = (pd.concat(fix_rows, ignore_index=True)
            .drop_duplicates(["strategy", "model", "attribute", "group", "metric"], keep="last"))

def headline(df):
    q = ("(attribute == 'overall' and metric in ['accuracy', 'auc']) or "
         "(attribute == 'race_eth_x_sex' and metric in ['DPD', 'EOD'])")
    return (df.query(q)
              .pivot(index=["model", "strategy"], columns="metric", values="value")
              [["accuracy", "auc", "DPD", "EOD"]].round(4))

headline(fix_df)

metric                  accuracy     auc     DPD     EOD
model   strategy                                        
LR      global_cutoff     0.8110  0.8831  0.5381  0.5327
        group_cutoffs     0.7846  0.8831  0.2268  0.1120
        none              0.8084  0.8831  0.5296  0.5778
        none_mitigated    0.7971  0.8006  0.3096  0.2118
XGBoost global_cutoff     0.8172  0.8926  0.5125  0.4824
        group_cutoffs     0.7909  0.8926  0.2149  0.1027
        none              0.8150  0.8926  0.5024  0.5077
        none_mitigated    0.8017  0.7899  0.2883  0.1841

### Finding: S2 is the fairest so far

| Model | Strategy | Accuracy | DPD | EOD |
|---|---|---|---|---|
| LR | none | 0.808 | 0.530 | 0.578 |
| LR | none_mitigated | 0.797 | 0.310 | 0.212 |
| LR | **group_cutoffs** | 0.785 | **0.227** | **0.112** |
| XGBoost | none | 0.815 | 0.502 | 0.508 |
| XGBoost | none_mitigated | 0.802 | 0.288 | 0.184 |
| XGBoost | **group_cutoffs** | 0.791 | **0.215** | **0.103** |

- **EOD is about half that of the 2014 mitigated models** (0.11 vs. 0.21 LR; 0.10 vs. 0.18 XGBoost). Differences of 0.08–0.10 are well above the noise level (about 0.04).
- DPD is also the lowest (about 0.22).
- **Cost in accuracy:** about 2.4 points vs. doing nothing, and about 1.2 points vs. the 2014 mitigated models.
- EOD is not 0 on 2017 because the thresholds were fitted on 2016, predictions are randomized, and some groups are small. It still carries over well to a new year.

## S3: Retrain on 2016

Start from a copy of each frozen pipeline (same settings, unfitted) and fit **both the preprocessor and the model** on 2016. This is the standard answer to drift: give the model the newest data. No fairness constraint is added.

In [12]:
from sklearn.base import clone

s3_fitted = {}
for name in ["lr", "xgb"]:
    label = MODEL_INFO[name]["model"]
    pipe = clone(models[name])
    t0 = time.perf_counter()
    pipe.fit(X16, y16)
    fit_seconds = time.perf_counter() - t0

    ev = evaluate_model(pipe, X17, y17, meta17)
    fix_rows.append(to_fix_rows(ev, "retrain", label, fit_seconds))
    s3_fitted[label] = pipe
    print(f"{label}: fit {fit_seconds:.1f}s")

LR: fit 8.6s
XGBoost: fit 4.8s


In [13]:
# 1. Frozen model untouched: should still give the S0 accuracy (0.8084)
print("frozen LR 2017 acc:", round((models["lr"].predict(X17) == y17).mean(), 4))

# 2. The new preprocessor really was fitted on 2016 (category lists differ from 2014)
for label, pipe in s3_fitted.items():
    n_new = len(pipe.named_steps["pre"].get_feature_names_out())
    print(f"{label}: {n_new} encoded features on 2016 (2014 had 456)")

# 3. Headline comparison
fix_df = (pd.concat(fix_rows, ignore_index=True)
            .drop_duplicates(["strategy", "model", "attribute", "group", "metric"], keep="last"))
headline(fix_df)

frozen LR 2017 acc: 0.8084
LR: 532 encoded features on 2016 (2014 had 456)
XGBoost: 532 encoded features on 2016 (2014 had 456)


metric                  accuracy     auc     DPD     EOD
model   strategy                                        
LR      global_cutoff     0.8110  0.8831  0.5381  0.5327
        group_cutoffs     0.7846  0.8831  0.2268  0.1120
        none              0.8084  0.8831  0.5296  0.5778
        none_mitigated    0.7971  0.8006  0.3096  0.2118
        retrain           0.8133  0.8849  0.5181  0.5117
XGBoost global_cutoff     0.8172  0.8926  0.5125  0.4824
        group_cutoffs     0.7909  0.8926  0.2149  0.1027
        none              0.8150  0.8926  0.5024  0.5077
        none_mitigated    0.8017  0.7899  0.2883  0.1841
        retrain           0.8197  0.8950  0.5046  0.4579

**Checks pass:**
- The frozen LR model still gives 0.8084 on 2017, so cloning did not touch it.
- The new preprocessor really was fitted on 2016: it has 532 encoded features vs. 456 in 2014.

### Finding: retraining ≈ S1

- **Accuracy is the best of any strategy** (0.813 LR, 0.820 XGBoost), but only 0.2–0.3 points above S1. AUC rises by just 0.002.
- **EOD (0.512 LR, 0.458 XGBoost) is within about 0.02 of S1,** back to roughly the 2014 level.

**Retraining restores the 2014 state, not fairness.** This fits Phase 3: the ranking was never broken, so a new model mostly re-learns the same thing with an updated threshold. A model trained only for accuracy reproduces the same group gaps.

**Confound:** the retrained models used all 131,089 rows of 2016, vs. 76,223 rows of 2014 training data (1.7×), so more occupation and birthplace categories passed the 30-row minimum (532 vs. 456 features). Even with more data, retraining only matched S1 on fairness, so the conclusion holds.

## S4: Retrain and re-mitigate on 2016

Repeat Phase 1's mitigation on 2016 data: `ExponentiatedGradient` with the equalized odds constraint on race/ethnicity × sex, using `fit_mitigated()` exactly as in notebook 01, starting from a copy of each unmitigated model's settings. Predictions are thresholded (vote share ≥ 0.5) as in Phase 1.

This is the most complete fix, and the most expensive one: LR takes about 9 minutes and XGBoost about 15.

In [14]:
from sklearn.base import clone
from src.models import fit_mitigated, SENSITIVE_COLS

s4_fitted = {}

def run_s4(name):
    label = MODEL_INFO[name]["model"]
    base_clf = clone(models[name].named_steps["model"])    # unfitted LR / XGBoost, same settings

    t0 = time.perf_counter()
    pipe = fit_mitigated(base_clf, X16, y16, X16[SENSITIVE_COLS])   # CHECK vs notebook 01
    fit_seconds = time.perf_counter() - t0

    ev = evaluate_model(pipe, X17, y17, meta17)
    fix_rows.append(to_fix_rows(ev, "retrain_mitigate", label, fit_seconds))
    s4_fitted[label] = pipe                                 # in memory only, not saved
    print(f"{label}: fit {fit_seconds/60:.1f} min")

In [15]:
run_s4("lr")

LR: fit 8.0 min


In [16]:
fix_df = (pd.concat(fix_rows, ignore_index=True)
            .drop_duplicates(["strategy", "model", "attribute", "group", "metric"], keep="last"))
headline(fix_df).loc["LR"]

metric,accuracy,auc,DPD,EOD
strategy,,,,
global_cutoff,0.8110,0.8831,0.5381,0.5327
group_cutoffs,0.7846,0.8831,0.2268,0.1120
none,0.8084,0.8831,0.5296,0.5778
none_mitigated,0.7971,0.8006,0.3096,0.2118
retrain,0.8133,0.8849,0.5181,0.5117
retrain_mitigate,0.7990,0.8139,0.3064,0.2602


In [17]:
run_s4("xgb")

XGBoost: fit 17.5 min


In [18]:
print("frozen LR 2017 acc:", round((models["lr"].predict(X17) == y17).mean(), 4))   # 0.8084
fix_df = (pd.concat(fix_rows, ignore_index=True)
            .drop_duplicates(["strategy", "model", "attribute", "group", "metric"], keep="last"))
print(fix_df.groupby(["strategy", "model"]).size())   # 12 combinations, 94 rows each
headline(fix_df)

frozen LR 2017 acc: 0.8084
strategy          model  
global_cutoff     LR         94
                  XGBoost    94
group_cutoffs     LR         94
                  XGBoost    94
none              LR         94
                  XGBoost    94
none_mitigated    LR         94
                  XGBoost    94
retrain           LR         94
                  XGBoost    94
retrain_mitigate  LR         94
                  XGBoost    94
dtype: int64


metric                    accuracy     auc     DPD     EOD
model   strategy                                          
LR      global_cutoff       0.8110  0.8831  0.5381  0.5327
        group_cutoffs       0.7846  0.8831  0.2268  0.1120
        none                0.8084  0.8831  0.5296  0.5778
        none_mitigated      0.7971  0.8006  0.3096  0.2118
        retrain             0.8133  0.8849  0.5181  0.5117
        retrain_mitigate    0.7990  0.8139  0.3064  0.2602
XGBoost global_cutoff       0.8172  0.8926  0.5125  0.4824
        group_cutoffs       0.7909  0.8926  0.2149  0.1027
        none                0.8150  0.8926  0.5024  0.5077
        none_mitigated      0.8017  0.7899  0.2883  0.1841
        retrain             0.8197  0.8950  0.5046  0.4579
        retrain_mitigate    0.8057  0.7986  0.2592  0.1098

### Finding: S4 works for XGBoost but not for LR

| Model | none_mitigated EOD | group_cutoffs EOD | **retrain_mitigate EOD** | retrain_mitigate accuracy |
|---|---|---|---|---|
| LR | 0.212 | 0.112 | **0.260** | 0.799 |
| XGBoost | 0.184 | 0.103 | **0.110** | 0.806 |

- **XGBoost:** EOD 0.110, about the same as S2 (within noise), with 1.5 points more accuracy, and 0.07 better than keeping the 2014 mitigated model.
- **LR:** EOD 0.260, *higher* than keeping the 2014 mitigated model (0.212). The table below shows why.

Mitigated AUC (0.81 / 0.80) comes from a vote share, not a probability, so it is not comparable with the other strategies' AUC.

In [19]:
from src.fairness import save_results

save_results(fix_df, FIX_PATH, keys=("strategy", "model"))

check = pd.read_csv(FIX_PATH)
print(check.shape)  
print(check.columns.tolist())
print(check.duplicated(["strategy", "model", "attribute", "group", "metric"]).sum())

(1128, 6)
['strategy', 'model', 'attribute', 'group', 'metric', 'value']
0


In [20]:
fix_df.query("metric == 'fit_seconds'").pivot(index="strategy", columns="model", values="value").round(1)

model,LR,XGBoost
strategy,,
global_cutoff,0.4,0.4
group_cutoffs,1.6,1.5
none,0.0,0.0
none_mitigated,0.0,0.0
retrain,8.6,4.8
retrain_mitigate,477.8,1052.8


### Cost: fitting time

| Strategy | LR | XGBoost |
|---|---|---|
| none / none_mitigated | 0 | 0 |
| global_cutoff (S1) | 0.4 s | 0.4 s |
| group_cutoffs (S2) | 1.6 s | 1.5 s |
| retrain (S3) | 8.6 s | 4.8 s |
| retrain_mitigate (S4) | 478 s (8.0 min) | 1,053 s (17.5 min) |

Times depend on the laptop, so the orders of magnitude matter more than the exact numbers: **S1 and S2 take seconds, S3 takes seconds to tens of seconds, and S4 takes minutes, about 300–600× S2.**

**Data cost is the same for S1–S4:** all need labelled 2016 data. Only S0 needs nothing new.

**Practical cost differs:**
- S2 uses sex and race/ethnicity to pick each person's threshold (explicit group-specific treatment, which is generally not allowed in US hiring or lending), and two identical people can get different answers because of randomization.
- S3 and S4 mean deploying and validating a new model.
- Note: every model here already uses sex and race/ethnicity as inputs, so "uses protected attributes" alone does not separate the strategies.

In [21]:
STRATS = ["none_mitigated", "retrain_mitigate"]

sub = fix_df.query("model == 'LR' and attribute == 'race_eth_x_sex' "
                   "and metric in ['TPR', 'FPR', 'Count'] and strategy in @STRATS")

tbl = sub.pivot(index="group", columns=["metric", "strategy"], values="value")
tbl = tbl[[("Count", "none_mitigated"),
           ("TPR", "none_mitigated"), ("TPR", "retrain_mitigate"),
           ("FPR", "none_mitigated"), ("FPR", "retrain_mitigate")]]
tbl = tbl.rename(columns={("Count", "none_mitigated"): ("Count", "2017")})

display(tbl.sort_values(("TPR", "retrain_mitigate")).round(3))

metric                     Count            TPR                   \
strategy          none_mitigated none_mitigated retrain_mitigate   
group                                                              
NH Other × Female         1239.0          0.546            0.516   
NH Black × Female         6511.0          0.638            0.578   
Hispanic × Male          23248.0          0.518            0.625   
NH Black × Male           5726.0          0.603            0.628   
NH White × Female        32246.0          0.604            0.658   
Hispanic × Female        19235.0          0.577            0.678   
NH White × Male          37500.0          0.645            0.705   
NH Asian × Female         3152.0          0.661            0.709   
NH Other × Male           1430.0          0.656            0.719   
NH Asian × Male           3593.0          0.730            0.777   

metric                       FPR                   
strategy          none_mitigated retrain_mitigate  
group                                              
NH Other × Female          0.102            0.085  
NH Black × Female          0.120            0.099  
Hispanic × Male            0.078            0.129  
NH Black × Male            0.115            0.125  
NH White × Female          0.119            0.139  
Hispanic × Female          0.076            0.113  
NH White × Male            0.133            0.171  
NH Asian × Female          0.106            0.119  
NH Other × Male            0.114            0.150  
NH Asian × Male            0.127            0.156

### Why is LR's S4 worse? One small group

**TPR decides EOD here** (FPR gaps are at most about 0.09). With re-mitigation, LR's lowest TPR belongs to **NH Other × Female: 1,239 people, TPR 0.516** (SE ≈ 0.03), while NH Asian × Male is highest at 0.777.

- **Without that one group,** the gap is 0.777 − 0.578 ≈ 0.20, about the same as keeping the 2014 mitigated model (0.21). So LR's S4 is roughly equal to `none_mitigated`, not worse.
- **Re-mitigation clearly helped the large groups Phase 3 flagged:** Hispanic men 0.518 → 0.625 and Hispanic women 0.577 → 0.678 (about 42,000 people together).
- **One real loss:** NH Black women 0.638 → 0.578 (6,511 people).

**General lesson:** EOD is the gap between the best and worst group, so one small group can decide it. Always look at the group table behind the number.

Likely reason the two models differ: predictions are thresholded vote shares, which keep a varying share of Fairlearn's guarantee (the guarantee is for its randomized predictions).

## Answer to RQ3: which strategy keeps the model fair most cheaply?

**Group cutoffs (S2) are the cheapest reliable way to restore fairness:** about half the EOD of the 2014 mitigated models, in seconds, with the same result for both model types, at a cost of about 2.4 accuracy points.

| | Fairness (EOD) | Accuracy | Time to fit | Works for both models? |
|---|---|---|---|---|
| S0 none | Worst (0.51–0.58) | Middle | 0 | — |
| S0 none_mitigated | Moderate (0.18–0.21) | Lower | 0 | Yes |
| S1 global cutoff | Poor (0.48–0.53) | Higher | < 1 s | Yes |
| **S2 group cutoffs** | **Best (0.10–0.11)** | Lowest | **~2 s** | **Yes** |
| S3 retrain | Poor (0.46–0.51) | Highest | 5–11 s | Yes |
| S4 retrain + re-mitigate | Best for XGBoost (0.11), not LR (0.26) | Lower | 9–16 min | **No** |

- **S1 and S3 only undo the drift.** They bring EOD back to the 2014 level, but the 2014 unfairness stays. Fixing accuracy does not fix fairness.
- **Keeping the 2014 mitigated model is free** and still gives moderate fairness three years later (Phase 2's finding).
- **S4 can match S2** (XGBoost: same EOD, 1.5 points more accuracy), but it costs minutes instead of seconds and was not reliable for LR.

**Why S2 over XGBoost's S4, which is as fair and more accurate?** S2 gives the same result for both models, is quick enough to refit every year, and needs no new model. But it uses group-specific thresholds and randomized decisions, which many real settings do not allow. **Where that is not allowed, retraining with mitigation (S4) on XGBoost is the better choice**, accepting the longer fit and checking the small groups every time.

**Limitations:**
- One fit year (2016) and one test year (2017); point estimates only, no bootstrap intervals.
- S2's decisions are randomized and explicitly group-based.
- S4 uses thresholded predictions, so Fairlearn's guarantee does not strictly apply.
- The retrained models had 1.7× more training rows than the 2014 models.
- Best case for every fix: it assumes 2016 labels are available in time.
- Mitigated AUC is not comparable with the other models' AUC.